In [1]:
import pandas as pd, numpy as np
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

d = pd.read_csv("../data/model_data.csv")
d["log_value"] = np.log(d["market_value_in_eur"])

drop = ["Player Name", "Club", "Nationality", "market_value_in_eur", "log_value",
        "days_old", "date_of_birth", "match_method", "club_mismatch",
        "age_group", "name_key", "club_key"]
X = d.drop(columns=[c for c in drop if c in d.columns])
X = pd.get_dummies(X, columns=["Position"])
X = X.select_dtypes(include=["number", "bool"]).astype(float)
y = d["log_value"]
print(X.shape)

(292, 47)


In [2]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)

models = {
    "baseline (predict the mean)": DummyRegressor(),
    "ridge regression": make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 20))),
    "random forest": RandomForestRegressor(n_estimators=300, min_samples_leaf=3, random_state=42),
}

for name, model in models.items():
    pred = cross_val_predict(model, X, y, cv=cv)
    rmse = np.sqrt(mean_squared_error(y, pred))
    print(f"{name:30s} RMSE {rmse:.3f} | R2 {r2_score(y, pred):.3f} | typical error ~{(np.exp(rmse)-1)*100:.0f}%")

baseline (predict the mean)    RMSE 0.964 | R2 -0.010 | typical error ~162%
ridge regression               RMSE 0.565 | R2 0.653 | typical error ~76%
random forest                  RMSE 0.597 | R2 0.613 | typical error ~82%


In [3]:
from sklearn.model_selection import RepeatedKFold, cross_val_score

rkf = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)

def rmse_cv(model, X):
    s = cross_val_score(model, X, y, cv=rkf, scoring="neg_root_mean_squared_error")
    return -s.mean(), s.std()

ridge = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 20)))
forest = RandomForestRegressor(n_estimators=200, min_samples_leaf=3, random_state=42)

pos_cols = [c for c in X.columns if c.startswith("Position_")]
age_only = X[["age"]]
age_pos = X[["age"] + pos_cols]
age_pos_sq = age_pos.assign(age_sq=X["age"] ** 2)
club_dummies = pd.get_dummies(d["Club"], prefix="Club").astype(float)

feature_sets = {
    "age only": age_only,
    "age + position": age_pos,
    "age + position + age squared": age_pos_sq,
    "all stats (current)": X,
    "all stats + club": pd.concat([X, club_dummies], axis=1),
}

for fs_name, Xs in feature_sets.items():
    r_mean, r_std = rmse_cv(ridge, Xs)
    f_mean, f_std = rmse_cv(forest, Xs)
    print(f"{fs_name:32s} ridge {r_mean:.3f} (±{r_std:.3f}) | forest {f_mean:.3f} (±{f_std:.3f})")

age only                         ridge 0.739 (±0.066) | forest 0.748 (±0.093)
age + position                   ridge 0.733 (±0.058) | forest 0.739 (±0.085)
age + position + age squared     ridge 0.676 (±0.068) | forest 0.739 (±0.085)
all stats (current)              ridge 0.562 (±0.049) | forest 0.594 (±0.062)
all stats + club                 ridge 0.478 (±0.041) | forest 0.588 (±0.063)


In [4]:
X_final = X.assign(age_sq=X["age"] ** 2)

ridge = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 20)))

cv = KFold(n_splits=5, shuffle=True, random_state=42)
d["pred_log"] = cross_val_predict(ridge, X_final, y, cv=cv)

d["pred_value"] = np.exp(d["pred_log"])
d["gap"] = d["pred_log"] - d["log_value"]        # positive = model says worth MORE than market

rmse = np.sqrt(mean_squared_error(y, d["pred_log"]))
print(f"RMSE {rmse:.3f} | R2 {r2_score(y, d['pred_log']):.3f}")

RMSE 0.534 | R2 0.691


In [5]:
cols = ["Player Name", "Club", "Position", "age", "market_value_in_eur", "pred_value", "gap"]
fmt = d[cols].copy()
fmt["market_value_in_eur"] = (fmt["market_value_in_eur"] / 1e6).round(1)
fmt["pred_value"] = (fmt["pred_value"] / 1e6).round(1)
fmt = fmt.rename(columns={"market_value_in_eur": "market_M", "pred_value": "model_M"})

print("Model says worth MORE than market (possible bargains):")
print(fmt.sort_values("gap", ascending=False).head(12).round(2).to_string(index=False))

print("\nModel says worth LESS than market (possible premiums):")
print(fmt.sort_values("gap").head(12).round(2).to_string(index=False))

Model says worth MORE than market (possible bargains):
     Player Name                    Club Position  age  market_M  model_M  gap
       Sam Morsy            Ipswich Town      MID 33.7       0.5      5.8 2.45
      James Bree             Southampton      DEF 27.5       2.5     14.5 1.76
   Rodrigo Muniz                  Fulham      FWD 24.1      20.0     88.7 1.49
Mateus Fernandes             Southampton      MID 20.9      15.0     63.6 1.44
 Luke Woolfenden            Ipswich Town      DEF 26.6       2.0      8.4 1.43
    Julio Enciso            Ipswich Town      MID 21.3      22.0     83.4 1.33
      Adam Smith             Bournemouth      DEF 34.1       0.8      2.8 1.33
Marshall Munetsi Wolverhampton Wanderers      MID 28.9      15.0     53.8 1.28
 Cameron Burgess            Ipswich Town      DEF 29.6       2.0      6.9 1.24
  Chris Richards          Crystal Palace      DEF 25.2      14.0     46.7 1.21
     Luke Thomas          Leicester City      DEF 24.0       6.0     19.3 1.

In [6]:
print(d.groupby("Club")["gap"].agg(["mean", "count"]).sort_values("mean", ascending=False).round(2))
print(d.groupby("Position")["gap"].mean().round(2))

d["age_band"] = pd.cut(d["age"], [18, 22, 25, 28, 31, 40])
print(d.groupby("age_band", observed=True)["gap"].agg(["mean", "count"]).round(2))

                         mean  count
Club                                
Southampton              0.58     15
Ipswich Town             0.47     15
Leicester City           0.31     13
Bournemouth              0.19     14
Wolverhampton Wanderers  0.18     15
Chelsea                  0.14     15
Brentford                0.14     13
Crystal Palace           0.10     14
Fulham                   0.08     15
Brighton & Hove Albion   0.07     16
Newcastle United         0.02     13
Everton                  0.01     17
Manchester City         -0.04     15
Nottingham Forest       -0.10     12
West Ham United         -0.13     14
Manchester United       -0.17     15
Tottenham Hotspur       -0.27     18
Liverpool               -0.29     13
Arsenal                 -0.37     15
Aston Villa             -0.45     15
Position
DEF    0.02
FWD    0.03
MID    0.02
Name: gap, dtype: float64
          mean  count
age_band             
(18, 22]  0.14     26
(22, 25]  0.06     73
(25, 28] -0.07     98
(28, 

In [7]:
d["gap_adj"] = d["gap"] - d.groupby("Club")["gap"].transform("mean")

cols = ["Player Name", "Club", "Position", "age", "market_M", "model_M", "gap", "gap_adj", "club_mismatch"]
d["market_M"] = (d["market_value_in_eur"] / 1e6).round(1)
d["model_M"] = (d["pred_value"] / 1e6).round(1)

print("Cheapest relative to squad-mates:")
print(d.sort_values("gap_adj", ascending=False)[cols].head(12).round(2).to_string(index=False))
print("\nMost expensive relative to squad-mates:")
print(d.sort_values("gap_adj")[cols].head(12).round(2).to_string(index=False))

Cheapest relative to squad-mates:
      Player Name                    Club Position  age  market_M  model_M  gap  gap_adj  club_mismatch
        Sam Morsy            Ipswich Town      MID 33.7       0.5      5.8 2.45     1.99          False
    Rodrigo Muniz                  Fulham      FWD 24.1      20.0     88.7 1.49     1.41          False
       James Bree             Southampton      DEF 27.5       2.5     14.5 1.76     1.18          False
       Adam Smith             Bournemouth      DEF 34.1       0.8      2.8 1.33     1.14          False
   Chris Richards          Crystal Palace      DEF 25.2      14.0     46.7 1.21     1.11          False
      Amad Diallo       Manchester United      MID 22.9      45.0    115.2 0.94     1.11          False
 Marshall Munetsi Wolverhampton Wanderers      MID 28.9      15.0     53.8 1.28     1.10          False
James Ward-Prowse         West Ham United      MID 30.6       8.0     20.5 0.94     1.07          False
  Luke Woolfenden            I

In [8]:
ridge.fit(X_final, y)
coef = pd.Series(ridge[-1].coef_, index=X_final.columns).sort_values()
print(coef.head(8).round(3))
print(coef.tail(8).round(3))

age_sq                    -2.242
Successful Passes_p90     -0.257
Clearances_p90            -0.118
Successful Crosses_p90    -0.098
Clearances Off Line_p90   -0.088
Appearances               -0.085
Blocks_p90                -0.083
Aerial Duels_p90          -0.083
dtype: float64
Successful fThird Passes_p90    0.156
aDuels Won_p90                  0.158
Shots_p90                       0.184
Passes_p90                      0.207
Clean Sheets_p90                0.208
Minutes                         0.283
Touches_p90                     0.300
age                             1.594
dtype: float64


In [9]:
ridge.fit(X_final, y)
ages = np.arange(19, 39)
rows = pd.DataFrame([X_final.mean()] * len(ages)).reset_index(drop=True)
rows["age"] = ages
rows["age_sq"] = ages ** 2
curve = np.exp(ridge.predict(rows[X_final.columns])) / 1e6
print(pd.Series(curve.round(1), index=ages, name="predicted value (M, same player stats)"))

19    42.1
20    42.3
21    41.6
22    40.0
23    37.7
24    34.7
25    31.3
26    27.7
27    23.9
28    20.2
29    16.7
30    13.5
31    10.7
32     8.3
33     6.3
34     4.7
35     3.4
36     2.4
37     1.7
38     1.2
Name: predicted value (M, same player stats), dtype: float64


In [10]:
from sklearn.linear_model import ElasticNetCV
from sklearn.ensemble import VotingRegressor

rkf = RepeatedKFold(n_splits=5, n_repeats=3, random_state=42)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

team_context = ["Minutes", "Appearances", "Clean Sheets_p90"]
X_noctx = X_final.drop(columns=team_context)

ridge = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 20)))
enet = make_pipeline(StandardScaler(), ElasticNetCV(l1_ratio=[0.2, 0.5, 0.8], cv=5, max_iter=20000))
forest = RandomForestRegressor(n_estimators=200, min_samples_leaf=3, random_state=42)
blend = VotingRegressor([("ridge", ridge), ("forest", forest)])

experiments = {
    "ridge, all features (current)": (ridge, X_final),
    "ridge, no team-context": (ridge, X_noctx),
    "elastic net, no team-context": (enet, X_noctx),
    "forest, no team-context": (forest, X_noctx),
    "ridge+forest blend, no team-context": (blend, X_noctx),
}

for name, (model, Xs) in experiments.items():
    s = cross_val_score(model, Xs, y, cv=rkf, scoring="neg_root_mean_squared_error")
    pred = cross_val_predict(model, Xs, y, cv=kf)
    spread = (pred - y).groupby(d["Club"]).mean().std()
    big = (np.exp(pred) > 100e6).sum()
    print(f"{name:38s} RMSE {-s.mean():.3f} (±{s.std():.3f}) | club spread {spread:.2f} | predicted >€100M: {big}")

print("actual players valued >€100M:", (d['market_value_in_eur'] > 100e6).sum())

ridge, all features (current)          RMSE 0.521 (±0.064) | club spread 0.26 | predicted >€100M: 8
ridge, no team-context                 RMSE 0.621 (±0.068) | club spread 0.36 | predicted >€100M: 5
elastic net, no team-context           RMSE 0.599 (±0.062) | club spread 0.38 | predicted >€100M: 1
forest, no team-context                RMSE 0.645 (±0.063) | club spread 0.43 | predicted >€100M: 0
ridge+forest blend, no team-context    RMSE 0.596 (±0.063) | club spread 0.39 | predicted >€100M: 0
actual players valued >€100M: 5


In [11]:
from sklearn.ensemble import VotingRegressor

rkf = RepeatedKFold(n_splits=5, n_repeats=3, random_state=42)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

team_context = ["Minutes", "Appearances", "Clean Sheets_p90"]
X_noctx = X_final.drop(columns=team_context)

X_hinge = X_noctx.drop(columns=["age_sq"]).copy()
X_hinge["age_over_28"] = (X_hinge["age"] - 28).clip(lower=0)
X_hinge["age_over_31"] = (X_hinge["age"] - 31).clip(lower=0)

ridge = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 20)))
forest = RandomForestRegressor(n_estimators=200, min_samples_leaf=3, random_state=42)
blend = VotingRegressor([("ridge", ridge), ("forest", forest)])

experiments = {
    "ridge, all features (current)": (ridge, X_final),
    "ridge, no team-context": (ridge, X_noctx),
    "ridge, no team-context + age hinges": (ridge, X_hinge),
    "forest, no team-context": (forest, X_noctx),
    "blend (ridge+forest), hinges": (blend, X_hinge),
}

old = d["age"] > 31
for name, (model, Xs) in experiments.items():
    s = cross_val_score(model, Xs, y, cv=rkf, scoring="neg_root_mean_squared_error")
    pred = cross_val_predict(model, Xs, y, cv=kf)
    spread = (pred - y).groupby(d["Club"]).mean().std()
    big = (np.exp(pred) > 100e6).sum()
    old_gap = (pred - y)[old].mean()
    print(f"{name:38s} RMSE {-s.mean():.3f} (±{s.std():.3f}) | club spread {spread:.2f} | >€100M: {big} | gap age>31: {old_gap:+.2f}")

print("actual players valued >€100M:", (d['market_value_in_eur'] > 100e6).sum())

ridge, all features (current)          RMSE 0.521 (±0.064) | club spread 0.26 | >€100M: 8 | gap age>31: +0.13
ridge, no team-context                 RMSE 0.621 (±0.068) | club spread 0.36 | >€100M: 5 | gap age>31: +0.25
ridge, no team-context + age hinges    RMSE 0.582 (±0.066) | club spread 0.37 | >€100M: 0 | gap age>31: +0.12
forest, no team-context                RMSE 0.645 (±0.063) | club spread 0.43 | >€100M: 0 | gap age>31: +0.14
blend (ridge+forest), hinges           RMSE 0.587 (±0.068) | club spread 0.40 | >€100M: 0 | gap age>31: +0.13
actual players valued >€100M: 5


In [12]:
X_hinge_all = X_final.drop(columns=["age_sq"]).copy()
X_hinge_all["age_over_28"] = (X_hinge_all["age"] - 28).clip(lower=0)
X_hinge_all["age_over_31"] = (X_hinge_all["age"] - 31).clip(lower=0)

experiments = {
    "A: all features + hinges": X_hinge_all,
    "B: drop Clean Sheets only + hinges": X_hinge_all.drop(columns=["Clean Sheets_p90"]),
    "C: drop all 3 team-context + hinges": X_hinge_all.drop(columns=team_context),
}

top = d["market_value_in_eur"].nlargest(5).index
for name, Xs in experiments.items():
    s = cross_val_score(ridge, Xs, y, cv=rkf, scoring="neg_root_mean_squared_error")
    pred = cross_val_predict(ridge, Xs, y, cv=kf)
    spread = (pred - y).groupby(d["Club"]).mean().std()
    old_gap = (pred - y)[old].mean()
    top_pred = (np.exp(pred[top]) / 1e6).round(0).tolist()
    print(f"{name:38s} RMSE {-s.mean():.3f} (±{s.std():.3f}) | club spread {spread:.2f} | gap age>31 {old_gap:+.2f} | top-5 predicted (M): {top_pred}")

print("top-5 actual (M):", (d["market_value_in_eur"].nlargest(5) / 1e6).tolist())

A: all features + hinges               RMSE 0.488 (±0.063) | club spread 0.27 | gap age>31 +0.08 | top-5 predicted (M): [68.0, 63.0, 52.0, 133.0, 81.0]
B: drop Clean Sheets only + hinges     RMSE 0.530 (±0.067) | club spread 0.35 | gap age>31 +0.08 | top-5 predicted (M): [64.0, 58.0, 47.0, 162.0, 87.0]
C: drop all 3 team-context + hinges    RMSE 0.582 (±0.066) | club spread 0.37 | gap age>31 +0.12 | top-5 predicted (M): [52.0, 76.0, 33.0, 91.0, 67.0]
top-5 actual (M): [180.0, 150.0, 120.0, 120.0, 120.0]


In [13]:
import os, joblib

XA = X_final.drop(columns=["age_sq"]).copy()
XA["age_over_28"] = (XA["age"] - 28).clip(lower=0)
XA["age_over_31"] = (XA["age"] - 31).clip(lower=0)

ridge = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 20)))

# every player is predicted by models that never saw him,
# averaged over 5 different splits so one lucky split doesn't decide
preds = []
for seed in range(5):
    kf = KFold(n_splits=5, shuffle=True, random_state=seed)
    preds.append(cross_val_predict(ridge, XA, y, cv=kf))
d["pred_log"] = np.mean(preds, axis=0)
d["pred_value"] = np.exp(d["pred_log"])
d["gap"] = d["pred_log"] - d["log_value"]
d["gap_adj"] = d["gap"] - d.groupby("Club")["gap"].transform("mean")
print(f"RMSE {np.sqrt(mean_squared_error(y, d['pred_log'])):.3f} | R2 {r2_score(y, d['pred_log']):.3f}")

# cautions: cases where the gap is not informative
flags = pd.DataFrame({
    "star_price": d["market_value_in_eur"] >= 60e6,
    "old_32plus": d["age"] >= 32,
    "teen_potential": d["age"] < 21,
    "price_floor": d["market_value_in_eur"] <= 1.5e6,
    "club_mismatch": d["club_mismatch"],
})
d["caution"] = flags.apply(lambda r: ", ".join(r.index[r]), axis=1)

# save the model, the predictions and the features
ridge.fit(XA, y)
os.makedirs("../models", exist_ok=True)
joblib.dump({"model": ridge, "columns": list(XA.columns)}, "../models/value_model.joblib")

d[["Player Name", "Club", "Position", "age", "Minutes", "market_value_in_eur",
   "pred_value", "gap", "gap_adj", "caution"]].to_csv("../data/predictions.csv", index=False)
XA.assign(**{"Player Name": d["Player Name"]}).to_csv("../data/features.csv", index=False)

# look at the results
d["market_M"] = (d["market_value_in_eur"] / 1e6).round(1)
d["model_M"] = (d["pred_value"] / 1e6).round(1)
cols = ["Player Name", "Club", "Position", "age", "market_M", "model_M", "gap_adj", "caution"]
print("\nCheapest relative to squad-mates:")
print(d.sort_values("gap_adj", ascending=False)[cols].head(10).round(2).to_string(index=False))
print("\nMost expensive relative to squad-mates:")
print(d.sort_values("gap_adj")[cols].head(10).round(2).to_string(index=False))
print("\nTop 5 by market value:")
print(d.nlargest(5, "market_value_in_eur")[["Player Name", "market_M", "model_M"]].to_string(index=False))

RMSE 0.484 | R2 0.746

Cheapest relative to squad-mates:
       Player Name              Club Position  age  market_M  model_M  gap_adj                 caution
         Sam Morsy      Ipswich Town      MID 33.7       0.5      4.3     1.75 old_32plus, price_floor
        James Bree       Southampton      DEF 27.5       2.5     14.3     1.15                        
        Adam Smith       Bournemouth      DEF 34.1       0.8      2.6     1.08 old_32plus, price_floor
   Luke Woolfenden      Ipswich Town      DEF 26.6       2.0      8.4     1.02                        
        Ryan Yates Nottingham Forest      MID 27.5      12.0     31.1     0.99                        
 James Ward-Prowse   West Ham United      MID 30.6       8.0     19.4     0.98                        
   Cameron Burgess      Ipswich Town      DEF 29.6       2.0      7.7     0.94                        
     Rodrigo Muniz            Fulham      FWD 24.1      20.0     51.8     0.91                        
  Tosin Adarabio

In [14]:
flags = flags.rename(columns={"teen_potential": "young_potential"})
flags["young_potential"] = d["age"] < 23
d["caution"] = flags.apply(lambda r: ", ".join(r.index[r]), axis=1)

d[["Player Name", "Club", "Position", "age", "Minutes", "market_value_in_eur",
   "pred_value", "gap", "gap_adj", "caution"]].to_csv("../data/predictions.csv", index=False)
print((d["caution"] != "").sum(), "players have at least one caution")

104 players have at least one caution
